# Kolokvijum II — Workflow i Task

## Tekst zadatka

> Napraviti konstruktore za objekte tipa **Workflow** i **Task**. Svaki Task opisan je nazivom, definicijom i metodom **execute**.
> Definicija predstavlja funkciju koja se izvršava pozivom funkcije execute. Rezultat poziva metode execute nad Task objektom
> je rezultat poziva funkcije date u atributu definicija. Metoda execute može primiti **proizvoljan broj argumenata**
> koje prosleđuje funkciji definisanoj u atributu definicija.
>
> Workflow sadrži podatke naziv, autor, spisak Task objekata i metodu execute. Workflow je **iterabilni objekat** —
> iterisanjem kroz workflow dobijaju se pojedinačni Task objekti. Za Workflow definisati operacije **map, flat, flatMap i filter**.
>
> Metoda execute izvršava sve zadate Task objekte tako što kao ulaz u execute metodu Task objekta dostavlja **rezultat prethodno
> izvršenog** Task objekta. Za ulaz u metodu za prvi task objekat uzimaju se **svi prosleđeni argumenti** iz metode execute u
> Workflow objektu. Povratna vrednost metode je rezultat izvršavanja **poslednjeg** Task objekta iz niza.
> Execute metode iz Workflow i Task objekta se izvršavaju **asinhrono**.
>
> Modifikovati Workflow objekat tako da je nad njim moguće primeniti **proizvoljnu kompoziciju transdjusera**.
> Demonstrirati ispravnost implementacije definisanjem **transdjusera za ispis svakog koraka izvršavanja** Workflow objekta.
>
> Demonstrirati ispravnost rešenja instanciranjem nekoliko Workflow i Task objekata i pozivanjem metoda definisanih nad njima.

## Podela na logičke jedinice

| Jedinica | Šta sadrži | Glavni mehanizam |
| --- | --- | --- |
| 1 | `Task` | funkcija kao podatak, `...args`, `async` metoda |
| 2 | alat: `collect`, `map`, `filter`, `compose` | transdjuseri i njihova kompozicija |
| 3 | `jeIterabilan`, `izravnaj`, `izvrsiRedom` | prepoznavanje iterabilnog, sekvencijalni `await` |
| 4 | `Workflow` | `Symbol.iterator` preko generatora, `map`/`filter`/`flat`/`flatMap`, `transduce` |
| 5 | transdjuser `saIspisom` | dekorisanje zadatka transdjuserom |
| 6 | demonstracija | ugnježđen workflow, kompozicija transdjusera |

Odluke koje tekst zadatka ne propisuje označene su sa **Pretpostavka** u odgovarajućoj jedinici.

## Pravila funkcionalne paradigme kojih se rešenje drži

| Pravilo | Kako je sprovedeno u ovom rešenju |
| --- | --- |
| **nepromenljivost** | `Object.freeze` nad svakim `Task`-om, nad spiskom zadataka i nad `Workflow`-om; `map`, `filter`, `flat`, `flatMap` i `primeni` vraćaju **nov** workflow |
| **bez `this` i bez `new`** | fabričke funkcije; sve metode su strelice nad zatvorenjem — jedini izuzetak je `[Symbol.iterator]`, koji jezik traži kao metodu |
| **objekat se gradi jednim izrazom** | `Object.freeze({ … })` — nema dopisivanja polja posle stvaranja |
| **funkcije su vrednosti** | `definicija` zadatka je običan atribut; transdjuseri se prosleđuju i spajaju kao podaci |
| **preklapanje umesto petlje** | `izvrsiRedom` je `reduce` preko lanca obećanja; u celom rešenju nema `for` petlje, brojača ni `let` promenljive |
| **kompozicija umesto proširivanja tipa** | nova obrada workflow-a je nov transdjuser, a ne nova metoda na `Workflow`-u |
| **razdvojena transformacija i oblik rezultata** | isti transdjuser sa `collect` daje niz, sa `(s, x) => s + x` broj |
| **efekti na ivici** | `console.log` postoji samo u demonstracijama i u transdjuserima čiji **jedini** posao jeste ispis odnosno merenje |

**Asinhronost je čuvana u tipu, ne u stanju.** `execute` uvek vraća `Promise`, pa se sekvenca gradi ulančavanjem
obećanja umesto promenljivom koja pamti dokle se stiglo.

**Nepotpuna čistoća, svesno i izolovano:** transdjuseri `saIspisom` i `merenje` postoje **zbog** efekta (ispis,
merenje vremena) — oni su omotači koji se dodaju kompozicijom i uklanjaju izostavljanjem, pa ostatak rešenja
ostaje bez efekata. `Date.now()` se javlja samo u `merenje`.

---
# Jedinica 1 · `Task`

In [ ]:
const Task = (naziv, definicija) => Object.freeze({
    naziv,
    definicija,                                                    // funkcija se čuva kao običan podatak

    execute: async (...argumenti) => definicija(...argumenti)      // proizvoljan broj argumenata; uvek vraća Promise
});

const saberi   = Task("saberi",   (a, b) => a + b);
const usporeno = Task("usporeno", async x => {               // definicija sme i sama da bude asinhrona
    await new Promise(kraj => setTimeout(kraj, 50));
    return x * 2;
});

console.log("naziv:", saberi.naziv, "| execute vraća:", saberi.execute(2, 3).constructor.name);
console.log("sinhrona definicija:", await saberi.execute(2, 3));
console.log("asinhrona definicija:", await usporeno.execute(21));

### Zašto ovaj mehanizam

**Funkcija kao atribut.** `definicija` je obična vrednost — funkcija se čuva, prosleđuje i zamenjuje kao broj ili string.
Zbog toga nije potrebna nijedna klasa po vrsti posla: „zadatak" je par imena i funkcije.

**`...argumenti` pa `definicija(...argumenti)`.** Zadatak traži da `execute` prihvati proizvoljan broj argumenata i
prosledi ih dalje. Sakupljanje u niz i rasipanje pri pozivu čini `execute` nezavisnim od toga koliko argumenata
definicija očekuje — isti kod radi za `(a, b) => a + b` i za `x => x * 2`.

**`async` metoda.** Zadatak traži asinhrono izvršavanje. Sa `async` povratna vrednost je **uvek** `Promise`, bez obzira
na to da li je definicija sinhrona ili asinhrona, pa pozivalac ima jedan jedini oblik poziva: `await task.execute(...)`.

**`Object.freeze`.** Zadatak je vrednost, ne promenljivo stanje — jednom napravljen, ne menja se; transformacije
(jedinica 5) prave **nov** zadatak.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| šta je „zadatak" | ime + funkcija | klasa koja implementira interfejs `Izvrsivo` |
| novi zadatak | nova funkcija na licu mesta | nova klasa ili anonimna implementacija |
| ulaz/izlaz | proizvoljan, `...args` | fiksiran potpis metode iz interfejsa |

**Prednosti.** Ponašanje se piše u jednom redu i prosleđuje kao argument, pa nema hijerarhije klasa za nešto što je
u suštini jedna funkcija (obrazac *Command* nestaje). Zadatak se lako pravi u petlji, iz konfiguracije ili iz drugog zadatka.

**Mane.** Ne postoji ugovor koji bi rekao koliko argumenata i kog tipa definicija očekuje — neslaganje se otkriva tek
pri izvršavanju, i to kao `NaN` ili `undefined`, a ne kao greška. Alatke i editor ne mogu da provere potpis,
a `async` sve greške pretvara u odbijena obećanja koja moraju da se hvataju.

---
# Jedinica 2 · Alat: transdjuseri

In [ ]:
const collect = (acc, e) => [...acc, e];                            // krajnji redjuser — rezultat je niz
const map     = f => next => (acc, e) => next(acc, f(e));           // transdjuser — preslikavanje
const filter  = p => next => (acc, e) => p(e) ? next(acc, e) : acc; // transdjuser — izdvajanje
const compose = (...fs) => x => fs.reduceRight((acc, f) => f(acc), x);

// smer: prvi argument compose-a prvi vidi element
console.log("compose:", [1, 2, 3, 4].reduce(compose(filter(x => x % 2 === 0), map(x => x * 10))(collect), []));
console.log("isti alat, drugi rezultat:", [1, 2, 3, 4].reduce(compose(filter(x => x % 2 === 0))((s, x) => s + x), 0));

### Zašto ovaj mehanizam

**Transdjuser je redjuser koji obavija drugi redjuser.** `map(f)` prima `next` (sledeći korak obrade) i vraća novi
redjuser koji prvo primeni `f`, pa pozove `next`. Zbog toga transformacija ne zna **odakle** podaci dolaze niti
**u šta** se skupljaju — isti `map(f)` radi nad nizom, nad workflowom ili nad tokom.

**`compose` umesto ulančavanja.** Kompozicija se pravi jednom i primenjuje bilo gde. Pošto se svi koraci izvršavaju
nad **istim elementom** pre prelaska na sledeći, ceo lanac je jedan prolaz i nema međunizova.

**Krajnji redjuser određuje oblik rezultata.** `collect` daje niz, `(s, x) => s + x` daje broj — transformacija ostaje ista.

Ovaj alat je ovde zato što četvrta jedinica traži da se nad `Workflow` objektom primeni **proizvoljna kompozicija
transdjusera**; bez njega bi `Workflow` morao da nudi zaseban metod za svaku vrstu izmene spiska zadataka.

### Funkcionalna paradigma prema OOP

**Prednosti.** Jedan mehanizam pokriva ono što u OOP rade obrasci *Decorator* (dodavanje ponašanja) i *Visitor*
(obilazak strukture): koraci se slažu kompozicijom, a ne nasleđivanjem. Nov korak ne traži izmenu ni `Workflow`-a ni
postojećih koraka.

**Mane.** Oblik `f => next => (acc, e) =>` je težak za čitanje dok se ne navikne, a greška u redosledu argumenata
`compose`-a daje pogrešan rezultat bez ijedne poruke. Kada nešto pukne unutar lanca, stek poziva je niz anonimnih
strelica, pa je izvor greške teže naći nego u petlji ili u metodi klase.

---
# Jedinica 3 · Pomoćne funkcije: izravnavanje i sekvencijalno izvršavanje

**Pretpostavka.** `flat` izravnava one članove spiska koji su **iterabilni** (ugnježđen `Workflow` ili niz zadataka).
Pojedinačan `Task` nije iterabilan, pa ostaje netaknut — time je razlika između „zadatak" i „grupa zadataka"
određena samim tipom, bez dodatnog polja.

In [ ]:
const jeIterabilan = x => x != null && typeof x[Symbol.iterator] === "function";

const izravnaj = zadaci => zadaci.flatMap(z => jeIterabilan(z) ? [...z] : [z]);

// zadaci se izvršavaju REDOM: svaki sledeći dobija rezultat prethodnog.
// Akumulator preklapanja je Promise koji nosi ULAZ sledećeg koraka (uvek niz argumenata).
const izvrsiRedom = (zadaci, argumenti) =>
    [...zadaci]
        .reduce((tok, zadatak) => tok
                    .then(ulaz => zadatak.execute(...ulaz))     // sledeći korak čeka prethodni
                    .then(rezultat => [rezultat]),              // rezultat prethodnog je jedini ulaz sledećeg
                Promise.resolve(argumenti))                     // prvi zadatak dobija SVE argumente
        .then(([poslednji]) => poslednji);                      // vrednost poslednjeg zadatka

const duplo = Task("duplo", x => x * 2);
const plusPet = Task("plusPet", x => x + 5);

console.log("izravnaj:", izravnaj([duplo, [plusPet, duplo], plusPet]).map(z => z.naziv));
console.log("redom (2+3, pa *2, pa +5):", await izvrsiRedom([saberi, duplo, plusPet], [2, 3]));
console.log("prazan spisak (propušta prvi argument):", await izvrsiRedom([], [1, 2]));

### Zašto ovaj mehanizam

**Preklapanje (`reduce`) preko lanca obećanja, a ne petlja.** Akumulator je `Promise` koji nosi **ulaz** sledećeg
koraka; svaki zadatak dopisuje po jedan `then`. Zato u funkciji nema nijedne promenljive koja se menja — sekvenca
je izražena kao vrednost koja nastaje spajanjem, a ne kao postupak koji nešto pamti između obrtaja.

**Zašto ne `Promise.all`.** Svaki zadatak zavisi od rezultata prethodnog, pa paralelno pokretanje nije moguće:
`all` bi pokrenuo sve odjednom, a nijedan osim prvog nema ulaz.

**`ulaz` je uvek niz, ne pojedinačna vrednost.** Prvi zadatak dobija **sve** argumente, svaki sledeći tačno jedan;
držanjem ulaza u nizu i rasipanjem sa `...ulaz` obe situacije se opisuju istim izrazom.

**Prazan spisak zadataka.** Preklapanje bez ijednog koraka vraća početnu vrednost, pa prazan workflow propušta
prvi prosleđeni argument — ponaša se kao neutralni element, bez ijedne posebne grane u kodu.

**Prepoznavanje iterabilnog.** `typeof x[Symbol.iterator] === "function"` je provera sposobnosti, a ne tipa —
radi za `Workflow`, za niz i za bilo šta drugo kroz šta se može proći `for...of` petljom.

**Funkcije su izvan `Workflow`-a.** Time `Workflow` u sledećoj jedinici može da se sastavi odjednom, u konačnom
obliku, a ove dve funkcije se testiraju same za sebe.

### Funkcionalna paradigma prema OOP

**Prednosti.** Tok podataka je vidljiv u samoj funkciji: ulaz ulazi, rezultat izlazi, nema polja koje neko usput menja.
Zato je `izvrsiRedom` upotrebljiva nad bilo kojim nizom objekata koji imaju `execute`, ne samo nad `Workflow`-om.

**Mane.** Sekvencijalno izvršavanje je najsporije moguće — nezavisni zadaci se ne mogu ubrzati bez menjanja modela.
Prva greška prekida ceo lanac i gubi već izračunate međurezultate, a lanac `then`-ova je teži za praćenje u debageru
od obične petlje. U OOP bi objekat mogao da pamti dokle je stigao i da nastavi posle greške; ovde bi to tražilo
dodatnu strukturu koja **nosi** stanje izvršavanja, umesto da ga krije u promenljivoj.

---
# Jedinica 4 · `Workflow`

In [ ]:
const Workflow = (naziv, autor, zadaci = []) => {
    const z = Object.freeze([...zadaci]);                                  // privatna, zamrznuta kopija spiska

    return Object.freeze({
        naziv, autor,

        get zadaci() { return [...z]; },                                   // izvedeni atribut — kopija ka spolja

        *[Symbol.iterator]() { yield* z; },                                // iterabilnost: for...of daje Task objekte

        map:     f => Workflow(naziv, autor, z.map(f)),                    // sve četiri vraćaju NOV Workflow
        filter:  p => Workflow(naziv, autor, z.filter(p)),
        flat:    () => Workflow(naziv, autor, izravnaj(z)),
        flatMap: f => Workflow(naziv, autor, izravnaj(z.map(f))),

        // proizvoljna kompozicija transdjusera nad spiskom zadataka
        transduce: (xform, krajnji = collect, pocetna = []) => z.reduce(xform(krajnji), pocetna),
        primeni: (...transdjuseri) => Workflow(naziv, autor, z.reduce(compose(...transdjuseri)(collect), [])),

        execute: (...argumenti) => izvrsiRedom(z, argumenti)                // rezultat poslednjeg zadatka
    });
};

const obrada = Workflow("obrada", "Blagoje", [duplo, plusPet]);

for (const z of obrada) console.log("  for...of:", z.naziv);
console.log("spread:", [...obrada].map(z => z.naziv), "| drugi obilazak:", [...obrada].length);
console.log("execute (3 -> *2 -> +5):", await obrada.execute(3));

console.log("map:   ", [...obrada.map(z => Task(z.naziv.toUpperCase(), z.definicija))].map(z => z.naziv));
console.log("filter:", [...obrada.filter(z => z.naziv !== "duplo")].map(z => z.naziv));
console.log("original netaknut:", [...obrada].map(z => z.naziv));

const ugnjezden = Workflow("glavni", "Blagoje", [saberi, obrada, plusPet]);
console.log("pre flat:", [...ugnjezden].map(z => jeIterabilan(z) ? `${z.naziv} (workflow)` : z.naziv));
console.log("posle flat:", [...ugnjezden.flat()].map(z => z.naziv));
console.log("flat pa execute (2+3 -> *2 -> +5 -> +5):", await ugnjezden.flat().execute(2, 3));

console.log("flatMap:", [...obrada.flatMap(z => Workflow(z.naziv, "Blagoje", [z, Task("kroz", x => x)]))].map(z => z.naziv));
console.log("transduce (samo nazivi):", obrada.transduce(map(z => z.naziv)));

### Zašto ovaj mehanizam

**Generator kao `[Symbol.iterator]`.** Zadatak traži da se iterisanjem dobijaju pojedinačni `Task` objekti.
`*[Symbol.iterator]() { yield* z; }` je cela implementacija: svaki poziv pravi **nov** generator, pa se workflow može
obići više puta, a uz iterabilnost stižu i `for...of`, `[...workflow]` i `yield*` u drugim generatorima.

**`map`, `filter`, `flat`, `flatMap` vraćaju `Workflow`, ne niz.** Tip je zatvoren nad svojim operacijama,
pa se pozivi ulančavaju (`w.filter(...).flat().map(...)`) i rezultat je i dalje izvršiv workflow.

**`transduce` i `primeni`.** Zahtev „proizvoljna kompozicija transdjusera" ispunjava se sa dva ulaza:
`transduce` vraća **rezultat** obrade (bilo kog oblika — niz, broj, tekst), a `primeni` vraća **nov Workflow**.
Nijedna izmena spiska zadataka ne traži novi metod na tipu — svaka se opisuje transdjuserom spolja.

**`get zadaci`.** Spisak je javni podatak iz zadatka, ali se izdaje kao kopija da spoljni kod ne bi menjao zamrznuti niz.

**`execute` je `async` i vraća rezultat poslednjeg zadatka**, jer sam posao radi `izvrsiRedom` iz prethodne jedinice.

### Funkcionalna paradigma prema OOP

| | Ovde (FP) | Kako bi bilo u OOP |
| --- | --- | --- |
| izmena spiska | `map`/`filter`/`primeni` vraćaju nov objekat | `workflow.addTask(...)` menja postojeći |
| obilazak | `Symbol.iterator`, radi sa svim jezičkim konstrukcijama | `getTasks()` pa petlja, ili *Iterator* obrazac |
| proširivanje | transdjuser spolja | nova metoda u klasi ili potklasa |

**Prednosti.** Workflow se ponaša kao ugrađena kolekcija: spread, destrukturiranje i `for...of` rade bez ijednog
dodatnog metoda. Pošto izmene prave nove objekte, isti workflow se može bezbedno deliti i granati u više varijanti
(sa ispisom, bez sporog koraka, samo prvi deo) bez rizika da jedna varijanta pokvari drugu.

**Mane.** Svaka operacija kopira spisak i pravi nov objekat — za dugačke workflow-e to je nepotrebna cena kada je
izmena jedna jedina. Iterabilnost skriva da `[...workflow]` svaki put gradi nov niz, pa se u petlji lako napravi
kvadratna složenost. U OOP bi jedna instanca kroz ceo život bila jeftinija i lakša za praćenje kroz debager.

---
# Jedinica 5 · Transdjuser za ispis svakog koraka

In [ ]:
// transdjuser: svaki Task zamenjuje NOVIM Task-om koji oko izvršavanja dopisuje ispis
const saIspisom = map(zadatak => Task(zadatak.naziv, async (...argumenti) => {
    console.log(`  → ${zadatak.naziv} ulaz:`, argumenti);
    const rezultat = await zadatak.execute(...argumenti);
    console.log(`  ← ${zadatak.naziv} izlaz:`, rezultat);
    return rezultat;
}));

const merenje = map(zadatak => Task(zadatak.naziv, async (...argumenti) => {
    const pocetak = Date.now();
    const rezultat = await zadatak.execute(...argumenti);
    console.log(`  ⏱ ${zadatak.naziv}: ${Date.now() - pocetak} ms`);
    return rezultat;
}));

const spor = Task("spor", async x => { await new Promise(k => setTimeout(k, 40)); return x + 1; });
const lanac = Workflow("lanac", "Blagoje", [duplo, spor, plusPet]);

console.log("bez ispisa:", await lanac.execute(3));

console.log("sa ispisom:");
console.log("rezultat:", await lanac.primeni(saIspisom).execute(3));

console.log("kompozicija: izbaci spor korak, pa ispiši ostale");
console.log("rezultat:", await lanac.primeni(filter(z => z.naziv !== "spor"), saIspisom).execute(3));

console.log("kompozicija: ispis + merenje vremena");
console.log("rezultat:", await lanac.primeni(saIspisom, merenje).execute(3));

console.log("original i dalje bez ispisa:", await lanac.execute(3));

### Zašto ovaj mehanizam

**Ispis je transdjuser, a ne dodatak u `execute`.** Da je `console.log` upisan u `izvrsiRedom`, ispis bi važio za
svako izvršavanje i isključivao bi se zastavicom. Ovako je ispis **jedan `map` transdjuser**: uključuje se tako što
se doda u kompoziciju, isključuje tako što se izostavi, i nijedan postojeći red koda se ne menja.

**Zamena zadatka novim zadatkom.** Transdjuser preslikava `Task` u **nov** `Task` čija definicija poziva stari
`execute` između dva ispisa. Original ostaje netaknut, pa isti `lanac` može istovremeno da postoji u varijanti sa
ispisom i bez njega.

**Kompozicija.** `primeni(filter(...), saIspisom)` pokazuje da se ispis slaže sa bilo kojim drugim transdjuserom:
prvi argument prvi vidi zadatak, pa se izbačeni koraci ni ne dekorišu. `merenje` je drugi takav omotač i dodaje se
istim putem, bez ijedne izmene u `saIspisom`.

### Funkcionalna paradigma prema OOP

**Prednosti.** Ovo je presečna odgovornost (ispis, merenje, ponavljanje, keširanje) rešena bez ijednog okvira:
u OOP bi to bio *Decorator* sa klasom po omotaču ili aspektno programiranje. Omotači se slažu u proizvoljnom
redosledu i ne znaju jedan za drugog.

**Mane.** Svaki sloj dodaje jedan nivo indirekcije: `await` unutar `await`-a, pa je stek pri grešci dublji i teže
čitljiv. Pošto dekorisani zadatak zadržava isti naziv, iz ispisa se ne vidi koliko je omotača u igri — a ako se isti
transdjuser slučajno primeni dvaput, ispis se udvostručuje bez ikakvog upozorenja.

---
# Jedinica 6 · Demonstracija

In [ ]:
const demo = async () => {
    console.log("═══ 1. Task ═══");
    const parsiraj   = Task("parsiraj",   tekst => tekst.split(",").map(Number));
    const bezNula    = Task("bezNula",    niz => niz.filter(x => x !== 0));
    const kvadrati   = Task("kvadrati",   niz => niz.map(x => x * x));
    const zbir       = Task("zbir",       niz => niz.reduce((s, x) => s + x, 0));
    const formatiraj = Task("formatiraj", x => `ukupno: ${x}`);
    const cekaj      = Task("cekaj",      async x => { await new Promise(k => setTimeout(k, 30)); return x; });

    console.log("execute sa dva argumenta:", await Task("spoji", (a, b) => `${a}-${b}`).execute("A", "B"));

    console.log("═══ 2. Workflow i iterabilnost ═══");
    const priprema = Workflow("priprema", "Blagoje", [parsiraj, bezNula]);
    const racun    = Workflow("racun",    "Blagoje", [kvadrati, cekaj, zbir]);
    const glavni   = Workflow("glavni",   "Blagoje", [priprema, racun, formatiraj]);

    for (const z of glavni) console.log("  član:", z.naziv);
    console.log("spisak:", glavni.zadaci.length, "| iterisanjem:", [...glavni].length);

    console.log("═══ 3. flat, flatMap, map, filter ═══");
    const ravan = glavni.flat();
    console.log("posle flat:", [...ravan].map(z => z.naziv));
    console.log("filter:", [...ravan.filter(z => z.naziv !== "cekaj")].map(z => z.naziv));
    console.log("map:", [...ravan.map(z => Task(z.naziv.toUpperCase(), z.definicija))].map(z => z.naziv));
    console.log("flatMap:", [...ravan.flatMap(z => Workflow("par", "Blagoje", [z, Task("kroz", x => x)]))].map(z => z.naziv));

    console.log("═══ 4. execute ═══");
    console.log("ugnježđen (flat pa execute):", await ravan.execute("1,0,2,3"));
    console.log("bez sporog koraka:", await ravan.filter(z => z.naziv !== "cekaj").execute("4,0,5"));
    console.log("prvi zadatak dobija sve argumente:",
                await Workflow("dva ulaza", "Blagoje", [Task("saberi", (a, b) => a + b), Task("puta10", x => x * 10)]).execute(2, 3));

    console.log("═══ 5. transdjuseri ═══");
    console.log("transduce -> nazivi:", ravan.transduce(map(z => z.naziv)));
    console.log("transduce -> broj zadataka:", ravan.transduce(map(() => 1), (s, x) => s + x, 0));
    console.log("primeni(saIspisom) — ispis svakog koraka:");
    console.log("rezultat:", await ravan.primeni(saIspisom).execute("1,0,2,3"));
    console.log("primeni(merenje) — trajanje svakog koraka:");
    console.log("rezultat:", await ravan.primeni(merenje).execute("1,0,2,3"));

    console.log("═══ 6. imutabilnost ═══");
    console.log("original netaknut:", [...glavni].map(z => z.naziv), "| rezultat i dalje:", await ravan.execute("1,0,2,3"));
};

await demo();

### Zašto ovaj mehanizam

Demonstracija ide redom kojim zadatak postavlja zahteve i svaki proverava **posmatranom posledicom**: da `execute`
prosleđuje sve argumente prvom zadatku, da se iterisanjem dobijaju `Task` objekti, da `flat` izravnava ugnježđen
workflow, da kompozicija transdjusera ispisuje svaki korak i da original posle svega ostaje nepromenjen.

Ugnježđen `glavni` workflow (dva workflow-a i jedan zadatak) postoji zato što `flat` i `flatMap` bez ugnježđenja
nemaju šta da pokažu — a `execute` nad neizravnatim spiskom ne bi radio, jer `Workflow` nema `execute` sa istim
značenjem kao `Task`.

### Funkcionalna paradigma prema OOP

**Prednosti.** Cela demonstracija je niz izraza čiji se rezultat odmah ispisuje — nema pripreme okruženja, lažnih
objekata ni redosleda koji utiče na ishod. Pošto original i izmenjena varijanta postoje istovremeno, imutabilnost
se dokazuje jednim poređenjem.

**Mane.** Sve provere su ručne: greška u lancu vidi se kao pogrešna vrednost na kraju, bez naznake u kom koraku je
nastala — zato transdjuser za ispis i postoji. Asinhronost dodatno otežava, jer se neuhvaćeno odbijeno obećanje
prijavljuje van mesta na kome je nastalo.